# TextBlob Coding Help!

## Import the required data bases

In [ ]:
import pandas as pd
import sqlalchemy as db

## Getting the files

In [ ]:
pmfeedback = pd.read_csv("https://github.com/casbdai/notebooks/raw/main/Module2/HypeCase/pmfeedback.csv")

In [ ]:
!wget https://github.com/casbdai/notebooks/raw/main/Module2/HypeCase/hypedb
engine = db.create_engine("sqlite:///hypedb")

In [ ]:
inspector = db.inspect(engine)
inspector.get_table_names()

## Applying LLM-Scoring

Run the following code to use a LLM for scoring:

In [ ]:
from openai import OpenAI
import json
from time import sleep

In [ ]:
client = OpenAI(api_key="ENTER YOUR API KEY HERE")

In [ ]:
model = "gpt-5.4"
# model = "gpt-4.1-mini"

# for gpt-5.4, select reasoning effort
reasoning_effort = "none" # alternatively try "low", "medium", "high"

# change your scoring dimension by adjusting the prompt
prompt = "Score the likelihood of a positive sentiment."


def score_text(text):
    try:
        request = {
            "model": model,
            "input": [
                {
                    "role": "system",
                    "content": (
                        prompt
                        + " Score the text with a likelihood between 0 (very low) and 1 (very high). "
                          "Return JSON only with the key 'score'."
                    ),
                },
                {
                    "role": "user",
                    "content": text[:400],
                },
            ],
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "score_response",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "score": {
                                "type": "number",
                                "minimum": 0,
                                "maximum": 1,
                            }
                        },
                        "required": ["score"],
                        "additionalProperties": False,
                    },
                    "strict": True,
                }
            },
        }

        # Only include reasoning for models you want to treat as GPT-5 reasoning models
        if model.startswith("gpt-5"):
            request["reasoning"] = {"effort": reasoning_effort}

        response = client.responses.create(**request)

        sleep(1)
        data = json.loads(response.output_text)
        return data["score"]

    except Exception as e:
        print("An unexpected error occurred:", e)
        return None

To apply the code follow the following syntax:


In [ ]:
# dataframe["new_column_with_sentiment"] = dataframe["column_to_analyze"].apply(score_text)**


As this code might take some time, you might want to track the progress. For this, use the tqdm library. It adds a .progress_apply function that helps you tracking how much time is needed to execute a function.

In [ ]:
from tqdm.auto import tqdm

tqdm.pandas()

# dataframe["new_column_with_sentiment"] = dataframe["column_to_analyze"].progress_apply(score_text)